<a href="https://colab.research.google.com/github/ishashahul/CertChain/blob/main/ner_spacy_mangalyaan_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Step 1: Import spaCy and Load a Language Model

First, we import the `spacy` library, which is a popular open-source library for advanced Natural Language Processing. We then load a pre-trained English language model, `en_core_web_sm`. This model is specifically designed for general-purpose NLP tasks, including Named Entity Recognition. Loading the model makes its linguistic data and statistical models available for processing text.

In [1]:
import spacy

# Load the English language model
nlp = spacy.load("en_core_web_sm")

### Step 2: Define the Text and Process it with spaCy

We define the paragraph on which we want to perform NER. Then, we pass this text to the `nlp` object (our loaded spaCy model). The `nlp()` function processes the input text and creates a `Doc` object. This `Doc` object is a container for all linguistic annotations, including tokens, sentences, parts-of-speech, and crucially, named entities.

In [2]:
paragraph = "The Mars Orbiter Mission (MOM), informally known as Mangalyaan, was launched into Earth orbit on 5 November 2013 by the Indian Space Research Organisation (ISRO) and entered Mars orbit on 24 September 2014. India became the first country to enter Mars orbit on its first attempt at a record cost of $74 million."

# Process the paragraph with the spaCy model
doc = nlp(paragraph)

### Step 3: Identify and Print Named Entities

The `Doc` object has an `.ents` property, which is a tuple of `Span` objects representing the named entities found in the text. We iterate through this collection. For each entity, we print its `text` (the actual words forming the entity), its `label_` (the type of entity, e.g., PERSON, ORG, DATE), and a human-readable explanation of that label using `spacy.explain(ent.label_)`. This step effectively extracts and presents all the named entities and their classifications.

In [3]:
print("Named Entities found in the paragraph:")
for ent in doc.ents:
    print(f"- Text: {ent.text}, Label: {ent.label_} ({spacy.explain(ent.label_)})")

Named Entities found in the paragraph:
- Text: The Mars Orbiter Mission, Label: ORG (Companies, agencies, institutions, etc.)
- Text: Mangalyaan, Label: PERSON (People, including fictional)
- Text: Earth, Label: LOC (Non-GPE locations, mountain ranges, bodies of water)
- Text: 5 November 2013, Label: DATE (Absolute or relative dates or periods)
- Text: the Indian Space Research Organisation, Label: ORG (Companies, agencies, institutions, etc.)
- Text: Mars, Label: LOC (Non-GPE locations, mountain ranges, bodies of water)
- Text: 24 September 2014, Label: DATE (Absolute or relative dates or periods)
- Text: India, Label: GPE (Countries, cities, states)
- Text: first, Label: ORDINAL ("first", "second", etc.)
- Text: Mars, Label: LOC (Non-GPE locations, mountain ranges, bodies of water)
- Text: first, Label: ORDINAL ("first", "second", etc.)
- Text: $74 million, Label: MONEY (Monetary values, including unit)


## Part 2: LSTM-based Word Generator Program

For the word generator, we'll use TensorFlow/Keras to build and train a simple LSTM model. First, we import the necessary libraries. We'll also define some sample text to train our model on, as training a robust language model requires a large corpus, which is beyond the scope of this demonstration. The goal here is to illustrate the pipeline.

In [4]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.utils import to_categorical
import numpy as np
import string

# Sample text data for training (simplified for demonstration)
# In a real-world scenario, this would be a much larger corpus.
corpus = "Natural Language Processing is a field of artificial intelligence that focuses on the interaction between computers and human language. It involves various tasks like text generation, translation, and sentiment analysis. Deep learning models, especially LSTMs, are very effective for sequence data such as text. Generating new words based on a seed is a common application of these models."

# Preprocessing: Convert to lowercase and remove punctuation
corpus = corpus.lower()
corpus = ''.join([char for char in corpus if char not in string.punctuation])

### Tokenization and Sequence Creation

We tokenize the text to convert words into numerical representations. Each unique word gets a unique integer ID. Then, we create input sequences: for each word in the text, we create a sequence of preceding words as input and the current word as the target. This prepares the data for training the LSTM to predict the next word in a sequence.

In [5]:
# Initialize and fit tokenizer
tokenizer = Tokenizer()
tokenizer.fit_on_texts([corpus])
total_words = len(tokenizer.word_index) + 1

# Create input sequences and target words
input_sequences = []
for line in corpus.split('. '):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

# Pad sequences to ensure uniform length
max_sequence_len = max([len(x) for x in input_sequences])
padded_sequences = np.array(tf.keras.preprocessing.sequence.pad_sequences(input_sequences,
                                                                      maxlen=max_sequence_len,
                                                                      padding='pre'))

# Split into features (X) and labels (y)
X, labels = padded_sequences[:,:-1], padded_sequences[:,-1]
y = to_categorical(labels, num_classes=total_words)

print(f"Total words (vocabulary size): {total_words}")
print(f"Maximum sequence length: {max_sequence_len}")
print(f"Number of training sequences: {len(X)}")

Total words (vocabulary size): 50
Maximum sequence length: 58
Number of training sequences: 57


### Model Architecture

We define a sequential Keras model with an Embedding layer, an LSTM layer, and a Dense output layer. The Embedding layer converts word indices into dense vectors. The LSTM layer processes sequences and captures long-term dependencies. The Dense output layer with a softmax activation predicts the probability distribution over the vocabulary for the next word.

In [6]:
embedding_dim = 100

model = Sequential()
model.add(Embedding(total_words, embedding_dim, input_length=max_sequence_len-1))
model.add(LSTM(150))
model.add(Dense(total_words, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

### Training the Model

We train the LSTM model using the prepared sequences. For this demonstration, we'll train for a small number of epochs. In practice, training a useful word generator requires significantly more data and epochs.

In [7]:
epochs = 50 # Reduced epochs for faster demonstration
history = model.fit(X, y, epochs=epochs, verbose=1)

Epoch 1/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 4s 127ms/step - accuracy: 0.0000e+00 - loss: 3.9142
Epoch 2/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step - accuracy: 0.1404 - loss: 3.9025
Epoch 3/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step - accuracy: 0.1754 - loss: 3.8923
Epoch 4/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 111ms/step - accuracy: 0.1930 - loss: 3.8803
Epoch 5/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 111ms/step - accuracy: 0.1930 - loss: 3.8609
Epoch 6/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step - accuracy: 0.0877 - loss: 3.8300
Epoch 7/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step - accuracy: 0.0702 - loss: 3.8047
Epoch 8/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step - accuracy: 0.1053 - loss: 3.7499
Epoch 9/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step - accuracy: 0.1930 - loss: 3.7195
Epoch 10/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step - accuracy: 0.1579 - loss: 3.6549
Epoch 11/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step - accuracy: 0.1228 - loss: 3.5937
Epoch 12/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 135ms/step - accuracy: 0.1404 

### Inference: Generating New Words

To generate new words, we take a seed text, tokenize and pad it to match the model's input length. The model then predicts the next word's probability distribution. We select the word with the highest probability, append it to the sequence, and repeat the process to generate multiple words. This loop continues for the desired number of words.

In [8]:
seed_text = "Natural Language Processing"
next_words = 5

for _ in range(next_words):
    token_list = tokenizer.texts_to_sequences([seed_text])[0]
    token_list = tf.keras.preprocessing.sequence.pad_sequences([token_list],
                                                                 maxlen=max_sequence_len-1,
                                                                 padding='pre')
    predicted_probs = model.predict(token_list, verbose=0)[0]
    predicted_word_index = np.argmax(predicted_probs)

    output_word = ""
    for word, index in tokenizer.word_index.items():
        if index == predicted_word_index:
            output_word = word
            break
    seed_text += " " + output_word

print(f"Generated text: {seed_text}")

Generated text: Natural Language Processing is a field of artificial


## Part 3: Healthcare NER Approach and Demonstration

### Proposed NER Approach for Healthcare Data

For extracting patient names, hospital names, dates of diagnosis, and monetary values from clinical reports, a **hybrid approach combining a fine-tuned spaCy model with rule-based patterns** is most suitable.

**Justification:**

1.  **Fine-tuned spaCy Model (Statistical/Machine Learning Approach):**
    *   **Pros:** SpaCy's pre-trained English models (`en_core_web_lg` or `en_core_web_trf`) provide a strong foundation. These models are good at identifying general entities like `PERSON`, `ORG`, `DATE`, and `MONEY`. Fine-tuning these models on a domain-specific dataset (clinical reports) labeled with the desired entities (Patient Name, Hospital Name, Date of Diagnosis, Monetary Value) would significantly improve accuracy. This approach is robust to variations in phrasing and can generalize well to unseen data after proper training.
    *   **Cons:** Requires a substantial amount of high-quality, human-annotated clinical data for effective fine-tuning, which can be time-consuming and expensive to obtain and label, especially given privacy concerns with patient data. Without sufficient data, performance might be suboptimal.

2.  **Rule-Based Approach (Alternative Comparison):**
    *   **Pros:** Does not require large labeled datasets. Rules can be defined based on patterns, keywords, or regular expressions (e.g., regex for dates, specific hospital names). It's highly interpretable and can be very accurate for specific, well-defined patterns.
    *   **Cons:** Extremely brittle and difficult to maintain. It fails to generalize to unseen variations and requires extensive manual effort to create and update rules. It struggles with ambiguity and context, which are prevalent in natural language. For instance, a rule for `DATE` might incorrectly tag a `DATE` within a drug name.

**Why Hybrid is Best:**
By fine-tuning a spaCy model, we leverage the power of statistical models to learn complex patterns and context, providing good generalization. We can then complement this with rule-based components (using spaCy's `Matcher` or custom pipeline components) to capture highly specific entities or to correct common errors that the statistical model might miss, especially for entities with very consistent formats (like specific diagnosis codes or currency formats that might not be consistently caught by the statistical model alone). This offers the best of both worlds: robust, generalizable performance from machine learning and precision for specific cases from rules, while mitigating the data dependency of a purely ML approach.

### Python Code Snippet using spaCy for Demonstration

Below is a demonstration using a standard spaCy model to extract entities that might align with the requested categories from a sample clinical sentence. Note that for precise and highly accurate extraction in a real clinical setting, fine-tuning on a specialized dataset would be critical, as the general `en_core_web_sm` model may not perfectly align entity labels to the specific healthcare domain requirements (e.g., distinguishing a general `ORG` from a `Hospital Name`).

In [9]:
import spacy

# Load the English language model
nlp = spacy.load("en_core_web_sm")

sample_clinical_sentence = "Patient John Doe was admitted to St. Jude's Hospital on 2023-10-26 with a diagnosis. The total bill was $15,000, and a follow-up is scheduled for next month."

# Process the sentence
doc = nlp(sample_clinical_sentence)

print("Extracted Entities from Clinical Sentence:")
patient_names = []
hospital_names = []
dates_of_diagnosis = []
monetary_values = []

for ent in doc.ents:
    # Map general spaCy entities to desired categories
    if ent.label_ == "PERSON":
        patient_names.append(ent.text)
    elif ent.label_ == "ORG": # ORG might represent hospitals
        hospital_names.append(ent.text)
    elif ent.label_ == "DATE":
        # Simple heuristic: if a DATE follows a hospital admission context, it could be diagnosis date
        # In a real system, more sophisticated context analysis or custom NER would be needed.
        if "admitted" in doc.text.lower() or "diagnosis" in doc.text.lower(): # Basic context check
             dates_of_diagnosis.append(ent.text)
        else:
            # If it's just a general date, add it as such for now
            dates_of_diagnosis.append(ent.text)
    elif ent.label_ == "MONEY":
        monetary_values.append(ent.text)

print(f"- Patient Names (PERSON): {patient_names if patient_names else 'Not found'}")
print(f"- Hospital Names (ORG): {hospital_names if hospital_names else 'Not found'}")
print(f"- Dates of Diagnosis (DATE): {dates_of_diagnosis if dates_of_diagnosis else 'Not found'}")
print(f"- Monetary Values (MONEY): {monetary_values if monetary_values else 'Not found'}")

Extracted Entities from Clinical Sentence:
- Patient Names (PERSON): ['John Doe']
- Hospital Names (ORG): Not found
- Dates of Diagnosis (DATE): ['2023-10-26', 'next month']
- Monetary Values (MONEY): ['15,000']


### Explanation of the Code Snippet

1.  **Import and Load Model:** We load the `en_core_web_sm` spaCy model, similar to Part 1.
2.  **Sample Clinical Sentence:** A representative sentence is defined to simulate a clinical report excerpt.
3.  **Process Sentence:** The `nlp()` function processes the sentence, generating a `Doc` object with linguistic annotations.
4.  **Iterate and Extract:** We loop through `doc.ents` to find recognized entities.
5.  **Categorization:** Based on spaCy's default entity labels (`PERSON`, `ORG`, `DATE`, `MONEY`), we categorize them into the requested types. It's important to note:
    *   `PERSON` directly maps to Patient Name.
    *   `ORG` is used for Hospital Name. While a general `ORG` might not always be a hospital, in this context, 'St. Jude's Hospital' is correctly identified as an `ORG`.
    *   `DATE` is used for Dates of Diagnosis. A simple keyword check (`"admitted"` or `"diagnosis"`) is added as a very basic context filter, but in a real system, more sophisticated rules or a custom entity for `DIAGNOSIS_DATE` would be needed.
    *   `MONEY` directly maps to Monetary Values.
6.  **Print Results:** The extracted entities are printed under their respective categories.